# Stacking Arabic BERT-based models

This notebook combines the positive-class probabilities produced by the four fine-tuned language models. A logistic-regression meta-model is trained on validation predictions and evaluated once on the held-out test set.

Run `03_train_base_models.ipynb` for each model before executing this notebook.

In [ ]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

from arabic_wsd.experiments import MODEL_CONFIGS, fit_logistic_stack

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
MODEL_KEYS = list(MODEL_CONFIGS)
MODEL_KEYS

## Load and align base-model predictions

Every file must contain predictions for the same examples in the same order. The label arrays are compared before any ensemble is fitted.

In [ ]:
def load_prediction_split(split_name):
    probabilities = {}
    reference_labels = None

    for model_key in MODEL_KEYS:
        path = ARTIFACTS_DIR / model_key / f'{split_name}_predictions.npz'
        archive = np.load(path)
        labels = archive['label']
        if reference_labels is None:
            reference_labels = labels
        elif not np.array_equal(reference_labels, labels):
            raise ValueError(f'Label alignment failed for {model_key} on {split_name}')
        probabilities[model_key] = archive['positive_probability']

    return probabilities, reference_labels

validation_probabilities, y_validation = load_prediction_split('validation')
test_probabilities, y_test = load_prediction_split('test')

## Compare model combinations

The experiment evaluates pairs, three-model combinations and the complete four-model ensemble. Each column of the meta-model input is the positive-class probability produced by one base model.

In [ ]:
rows = []
for ensemble_size in range(2, len(MODEL_KEYS) + 1):
    for selected_models in combinations(MODEL_KEYS, ensemble_size):
        validation_features = np.column_stack([validation_probabilities[name] for name in selected_models])
        test_features = np.column_stack([test_probabilities[name] for name in selected_models])
        _, metrics = fit_logistic_stack(
            validation_features,
            y_validation,
            test_features,
            y_test,
            seed=9,
        )
        rows.append({'models': ' + '.join(selected_models), **metrics})

ensemble_results = pd.DataFrame(rows).sort_values('f1', ascending=False)
ensemble_results.style.format({column: '{:.2%}' for column in ['accuracy', 'f1', 'precision', 'recall']})

## Results reported in the publication

The best published configuration combined AraBERT and ARBERTv2. The two-model ensemble outperformed the larger combinations, showing that complementarity mattered more than ensemble size.

In [ ]:
reported_results = pd.DataFrame([
    {'models': 'AraBERT + CAMeLBERT', 'accuracy': 0.8161, 'f1': 0.8241, 'precision': 0.7911, 'recall': 0.8599},
    {'models': 'AraBERT + ARBERTv2', 'accuracy': 0.8211, 'f1': 0.8297, 'precision': 0.7935, 'recall': 0.8693},
    {'models': 'AraBERT + CAMeLBERT + ARBERTv2', 'accuracy': 0.8188, 'f1': 0.8269, 'precision': 0.7930, 'recall': 0.8638},
    {'models': 'All four models', 'accuracy': 0.8194, 'f1': 0.8276, 'precision': 0.7932, 'recall': 0.8651},
])
reported_results.style.format({column: '{:.2%}' for column in ['accuracy', 'f1', 'precision', 'recall']})

Results summarized from:

Djaidri, A., Aliane, H. and Azzoune, H. (2025). “Enhancing Arabic Word Sense Disambiguation with Ensemble BERT-Based Models.” *Arabic Language Processing: From Theory to Practice*, CCIS 2339, 261–272. [https://doi.org/10.1007/978-3-031-79164-2_23](https://doi.org/10.1007/978-3-031-79164-2_23).